In [ ]:
import scanpy as sc
import scvi
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os

# 1. Configuration & Paths
sc.settings.verbosity = 3
FILE_PATH = Path(os.getenv("SC_INTEGRATED_H5AD", "./data/processed/adata_integrated_cleaned.h5ad"))
FIG_DIR = Path(os.getenv("SC_FIG_DIR", "./results/figures"))
PLOT_DIR = Path(os.getenv("SC_PLOT_DIR", "./results/plots"))
MARKERS_CSV = Path(os.getenv("SC_MARKERS_CSV", "./results/markers_scvi_leiden_filtered.csv"))

sc.settings.figdir = str(FIG_DIR)
os.makedirs(sc.settings.figdir, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# 2. Load Data & Run Clustering
adata = sc.read_h5ad(FILE_PATH)
print(adata)
print(adata.obs.head())

sc.tl.leiden(adata, resolution=1)
sc.tl.rank_genes_groups(adata, 'leiden')

# Save differential expression plots
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, save=".rank_genes_groups.png")

# 3. Filter Markers via Scanpy Wilcoxon Results
markers = sc.get.rank_genes_groups_df(adata, None)
markers = markers[(markers.pvals_adj < 0.05) & (markers.logfoldchanges > 0.5)]
print(markers)

# 4. Load Pre-calculated scVI Differential Markers
markers_scvi = pd.read_csv(MARKERS_CSV)
print(markers_scvi.head())
print(adata.obs.head())

# 5. Dimension Reduction UMAP Visualization
sc.settings.set_figure_params(figsize=(5, 5), fontsize=8)
sc.pl.umap(adata, color=['leiden'], frameon=False, legend_loc='on data', size=1.2)
plt.show()

# Gene Expression Mapping on UMAP
sc.settings.set_figure_params(figsize=(6, 6), fontsize=8)
sc.pl.umap(adata, color=['HIST1H4C', 'H2AFZ'], frameon=False, layer='scvi_normalized', vmax=5, size=1.3)

# ==============================================================================
# [User Notebook Reference Notes: Cluster Identities & Markers]
# ==============================================================================
# Cluster 0:  DACH1, RP11-32K4.1, MAP2, SLIT2, SOX2, TCF7L2, WNT5B; SOX2
# Cluster 1:  HMGN2, HIST1H4C, H2AFZ, TUBA1B, TUBB, DACH1, COL4A6
# Cluster 2:  TOP2A, SOX2 -> Non-telencephalic NPC
# Cluster 4:  DLK1, TUBA1B
# Cluster 5:  TUBA1B
# Cluster 7:  neural_crest: SOX10, ROBO1, PAX3, LMO4, CCSER1, IFIT2
# Cluster 9:  DLK1
# Cluster 10: TCF7L2 -> NPC
# Cluster 11: ENO2
# Cluster 13: diencephalic_excitatory_neuron_CP: PITX2, TTR, EOMES / PSC POU5F1
# Cluster 14: HERC5, IFIT2
# Cluster 15: neuron: DCX, EBF1, UNC5C
# Cluster 16: neuron marker: DCX, STMN2, TUBB3 -> Ventral Telencephalic Neuron
# Cluster 18: choroid_plexus_epithelium: PBFOX1
# Cluster 20: DACH1, RP11-32K4.1, RP11-499F3.2, COL4A5, COL4A6, SLIT2, SOX6, WNT5B -> NPC
# ==============================================================================

# 6. Specific Cluster Marker Exploration
cluster_markers = markers[markers["group"] == "2"]  
cluster_markers_sorted = cluster_markers.sort_values(by="scores", ascending=False)
print("Top 15 markers for Cluster 2:")
print(cluster_markers_sorted.head(15))

print("Columns in markers_scvi:", markers_scvi.columns)
print("Querying SOX2 across groups:")
print(markers[markers.names == 'SOX2'])

print("Top markers for Cluster 10:")
print(markers[markers.group == 10].head())

# 7. Dotplot Visualization for Publication Preparation
sc.settings.figdir = str(PLOT_DIR)
sc.settings.set_figure_params(dpi=100, dpi_save=300, frameon=False)
sc.pl.rank_genes_groups_dotplot(
    adata, 
    groupby="leiden", 
    standard_scale="var", 
    n_genes=5,
    save=".leiden_marker_dotplot.png"
)